In [3]:
import os
os.rename("/content/orders (1).csv", "/content/orders.csv")

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("ECommerceAnalytics").getOrCreate()
sc = spark.sparkContext
customers   = spark.read.csv("customers.csv",   header=True, inferSchema=True)
products    = spark.read.csv("products.csv",    header=True, inferSchema=True)
orders      = spark.read.csv("orders.csv",      header=True, inferSchema=True)
order_items = spark.read.csv("order_items.csv", header=True, inferSchema=True)
payments    = spark.read.csv("payments.csv",    header=True, inferSchema=True)

**SECTION 1 — DataFrame Basics**

In [6]:
#1
customers.show()


+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [7]:
#2
customers.printSchema()


root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- signup_date: date (nullable = true)



In [8]:
#3
print(customers.count())


15


In [9]:
#4
customers.show(5)

+-----------+-----+---------+---+-----------+
|customer_id| name|     city|age|signup_date|
+-----------+-----+---------+---+-----------+
|          1| Amit|Hyderabad| 28| 2023-01-10|
|          2|Priya|Bangalore| 32| 2023-02-12|
|          3|Rahul|   Mumbai| 29| 2023-03-14|
|          4|Sneha|    Delhi| 35| 2023-04-15|
|          5|Arjun|  Chennai| 27| 2023-05-11|
+-----------+-----+---------+---+-----------+
only showing top 5 rows


In [10]:
#5
customers.select("name", "city").show()

+------+---------+
|  name|     city|
+------+---------+
|  Amit|Hyderabad|
| Priya|Bangalore|
| Rahul|   Mumbai|
| Sneha|    Delhi|
| Arjun|  Chennai|
| Meera|Hyderabad|
| Karan|     Pune|
|  Neha|    Delhi|
| Divya|Bangalore|
|Vikram|   Mumbai|
|  Ritu|Hyderabad|
|Sanjay|    Delhi|
|Naveen|  Chennai|
|Farhan|   Mumbai|
|Simran|Bangalore|
+------+---------+



In [11]:
#6
print(products.count())

10


In [12]:
#7
products.select("product_name", "price").show()

+------------+-----+
|product_name|price|
+------------+-----+
|      Laptop|75000|
|  Headphones| 3000|
|    Keyboard| 1500|
|     Monitor|12000|
|Office Chair| 7000|
|        Desk|15000|
|  Smartphone|40000|
|    Notebook|  100|
|         Pen|   20|
|      Tablet|30000|
+------------+-----+



In [13]:
#8
orders.show()

+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|       1|          1|2024-03-01|Delivered|
|       2|          2|2024-03-02|Delivered|
|       3|          3|2024-03-03|Cancelled|
|       4|          4|2024-03-04|Delivered|
|       5|          5|2024-03-05|Delivered|
|       6|          6|2024-03-06|Delivered|
|       7|          7|2024-03-07|  Pending|
|       8|          8|2024-03-08|Delivered|
|       9|          9|2024-03-09|Delivered|
|      10|         10|2024-03-10|Delivered|
+--------+-----------+----------+---------+



In [14]:
#9
print(orders.count())

10


In [15]:
#10
payments.show()


+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|         1|       1| Credit Card| 81000|
|         2|       2|         UPI|  1500|
|         3|       3|  Debit Card| 75000|
|         4|       4| Credit Card| 12000|
|         5|       5|         UPI| 40000|
|         6|       6|         UPI|  9000|
|         7|       7|  Debit Card|  3000|
|         8|       8| Credit Card| 30000|
|         9|       9|         UPI|   500|
|        10|      10|         UPI|   200|
+----------+--------+------------+------+



**SECTION 2 — Filtering Data**

In [17]:
#11
customers.filter(F.col("city") == "Hyderabad").show()

+-----------+-----+---------+---+-----------+
|customer_id| name|     city|age|signup_date|
+-----------+-----+---------+---+-----------+
|          1| Amit|Hyderabad| 28| 2023-01-10|
|          6|Meera|Hyderabad| 31| 2023-06-10|
|         11| Ritu|Hyderabad| 34| 2023-08-10|
+-----------+-----+---------+---+-----------+



In [18]:
#12
customers.filter(F.col("age") > 30).show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          2| Priya|Bangalore| 32| 2023-02-12|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
+-----------+------+---------+---+-----------+



In [19]:
#13
products.filter(F.col("price") > 10000).show()


+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       104|     Monitor|Electronics|12000|
|       106|        Desk|  Furniture|15000|
|       107|  Smartphone|Electronics|40000|
|       110|      Tablet|Electronics|30000|
+----------+------------+-----------+-----+



In [20]:
#14
products.filter(F.col("category") == "Electronics").show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       102|  Headphones|Electronics| 3000|
|       103|    Keyboard|Electronics| 1500|
|       104|     Monitor|Electronics|12000|
|       107|  Smartphone|Electronics|40000|
|       110|      Tablet|Electronics|30000|
+----------+------------+-----------+-----+



In [21]:
#15
orders.filter(F.col("status") == "Delivered").show()

+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|       1|          1|2024-03-01|Delivered|
|       2|          2|2024-03-02|Delivered|
|       4|          4|2024-03-04|Delivered|
|       5|          5|2024-03-05|Delivered|
|       6|          6|2024-03-06|Delivered|
|       8|          8|2024-03-08|Delivered|
|       9|          9|2024-03-09|Delivered|
|      10|         10|2024-03-10|Delivered|
+--------+-----------+----------+---------+



In [22]:
#16
orders.filter(F.col("status") == "Cancelled").show()


+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|       3|          3|2024-03-03|Cancelled|
+--------+-----------+----------+---------+



In [23]:
#17
payments.filter(F.col("payment_type") == "UPI").show()


+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|         2|       2|         UPI|  1500|
|         5|       5|         UPI| 40000|
|         6|       6|         UPI|  9000|
|         9|       9|         UPI|   500|
|        10|      10|         UPI|   200|
+----------+--------+------------+------+



In [24]:
#18
customers.filter(F.col("city").isin("Bangalore", "Delhi")).show()


+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          2| Priya|Bangalore| 32| 2023-02-12|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [25]:
#19
products.filter(F.col("price") < 1000).show()


+----------+------------+----------+-----+
|product_id|product_name|  category|price|
+----------+------------+----------+-----+
|       108|    Notebook|Stationery|  100|
|       109|         Pen|Stationery|   20|
+----------+------------+----------+-----+



In [26]:
#20
customers.filter(F.col("age").between(25, 35)).show()


+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



**SECTION 3 — Selecting and Renaming**

In [27]:
#21
products.withColumnRenamed("price","product_price").show()


+----------+------------+-----------+-------------+
|product_id|product_name|   category|product_price|
+----------+------------+-----------+-------------+
|       101|      Laptop|Electronics|        75000|
|       102|  Headphones|Electronics|         3000|
|       103|    Keyboard|Electronics|         1500|
|       104|     Monitor|Electronics|        12000|
|       105|Office Chair|  Furniture|         7000|
|       106|        Desk|  Furniture|        15000|
|       107|  Smartphone|Electronics|        40000|
|       108|    Notebook| Stationery|          100|
|       109|         Pen| Stationery|           20|
|       110|      Tablet|Electronics|        30000|
+----------+------------+-----------+-------------+



In [28]:
#22
customers.withColumnRenamed("name", "customer_name").show()


+-----------+-------------+---------+---+-----------+
|customer_id|customer_name|     city|age|signup_date|
+-----------+-------------+---------+---+-----------+
|          1|         Amit|Hyderabad| 28| 2023-01-10|
|          2|        Priya|Bangalore| 32| 2023-02-12|
|          3|        Rahul|   Mumbai| 29| 2023-03-14|
|          4|        Sneha|    Delhi| 35| 2023-04-15|
|          5|        Arjun|  Chennai| 27| 2023-05-11|
|          6|        Meera|Hyderabad| 31| 2023-06-10|
|          7|        Karan|     Pune| 33| 2023-06-22|
|          8|         Neha|    Delhi| 30| 2023-07-10|
|          9|        Divya|Bangalore| 26| 2023-07-15|
|         10|       Vikram|   Mumbai| 40| 2023-08-01|
|         11|         Ritu|Hyderabad| 34| 2023-08-10|
|         12|       Sanjay|    Delhi| 38| 2023-08-21|
|         13|       Naveen|  Chennai| 28| 2023-09-01|
|         14|       Farhan|   Mumbai| 36| 2023-09-10|
|         15|       Simran|Bangalore| 25| 2023-09-18|
+-----------+-------------+-

In [29]:
#23
customers.select("name", "age").show()


+------+---+
|  name|age|
+------+---+
|  Amit| 28|
| Priya| 32|
| Rahul| 29|
| Sneha| 35|
| Arjun| 27|
| Meera| 31|
| Karan| 33|
|  Neha| 30|
| Divya| 26|
|Vikram| 40|
|  Ritu| 34|
|Sanjay| 38|
|Naveen| 28|
|Farhan| 36|
|Simran| 25|
+------+---+



In [30]:
#24
products.select("product_name", "category").show()


+------------+-----------+
|product_name|   category|
+------------+-----------+
|      Laptop|Electronics|
|  Headphones|Electronics|
|    Keyboard|Electronics|
|     Monitor|Electronics|
|Office Chair|  Furniture|
|        Desk|  Furniture|
|  Smartphone|Electronics|
|    Notebook| Stationery|
|         Pen| Stationery|
|      Tablet|Electronics|
+------------+-----------+



In [31]:
#25
orders.select("order_id", "order_date").show()


+--------+----------+
|order_id|order_date|
+--------+----------+
|       1|2024-03-01|
|       2|2024-03-02|
|       3|2024-03-03|
|       4|2024-03-04|
|       5|2024-03-05|
|       6|2024-03-06|
|       7|2024-03-07|
|       8|2024-03-08|
|       9|2024-03-09|
|      10|2024-03-10|
+--------+----------+



In [32]:
#26
payments.select("payment_type", "amount").show()


+------------+------+
|payment_type|amount|
+------------+------+
| Credit Card| 81000|
|         UPI|  1500|
|  Debit Card| 75000|
| Credit Card| 12000|
|         UPI| 40000|
|         UPI|  9000|
|  Debit Card|  3000|
| Credit Card| 30000|
|         UPI|   500|
|         UPI|   200|
+------------+------+



In [33]:
#27
products.withColumn("price_in_lakhs", F.col("price") / 100000).show()


+----------+------------+-----------+-----+--------------+
|product_id|product_name|   category|price|price_in_lakhs|
+----------+------------+-----------+-----+--------------+
|       101|      Laptop|Electronics|75000|          0.75|
|       102|  Headphones|Electronics| 3000|          0.03|
|       103|    Keyboard|Electronics| 1500|         0.015|
|       104|     Monitor|Electronics|12000|          0.12|
|       105|Office Chair|  Furniture| 7000|          0.07|
|       106|        Desk|  Furniture|15000|          0.15|
|       107|  Smartphone|Electronics|40000|           0.4|
|       108|    Notebook| Stationery|  100|         0.001|
|       109|         Pen| Stationery|   20|        2.0E-4|
|       110|      Tablet|Electronics|30000|           0.3|
+----------+------------+-----------+-----+--------------+



In [34]:
#28
customers.withColumn("age_group",F.when(F.col("age") < 30, "Young").otherwise("Senior")).show()

+-----------+------+---------+---+-----------+---------+
|customer_id|  name|     city|age|signup_date|age_group|
+-----------+------+---------+---+-----------+---------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|    Young|
|          2| Priya|Bangalore| 32| 2023-02-12|   Senior|
|          3| Rahul|   Mumbai| 29| 2023-03-14|    Young|
|          4| Sneha|    Delhi| 35| 2023-04-15|   Senior|
|          5| Arjun|  Chennai| 27| 2023-05-11|    Young|
|          6| Meera|Hyderabad| 31| 2023-06-10|   Senior|
|          7| Karan|     Pune| 33| 2023-06-22|   Senior|
|          8|  Neha|    Delhi| 30| 2023-07-10|   Senior|
|          9| Divya|Bangalore| 26| 2023-07-15|    Young|
|         10|Vikram|   Mumbai| 40| 2023-08-01|   Senior|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|   Senior|
|         12|Sanjay|    Delhi| 38| 2023-08-21|   Senior|
|         13|Naveen|  Chennai| 28| 2023-09-01|    Young|
|         14|Farhan|   Mumbai| 36| 2023-09-10|   Senior|
|         15|Simran|Bangalore| 

In [35]:
#29
order_items.filter(F.col("quantity") > 2).show()


+--------+----------+--------+
|order_id|product_id|quantity|
+--------+----------+--------+
|       6|       102|       3|
|       9|       108|       5|
|      10|       109|      10|
+--------+----------+--------+



In [36]:
#30
customers.orderBy("age").show()


+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|         15|Simran|Bangalore| 25| 2023-09-18|
|          9| Divya|Bangalore| 26| 2023-07-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          7| Karan|     Pune| 33| 2023-06-22|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
+-----------+------+---------+---+-----------+



**SECTION 4 — Sorting and Limiting**

In [37]:
#31
customers.orderBy(F.col("age").asc()).show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|         15|Simran|Bangalore| 25| 2023-09-18|
|          9| Divya|Bangalore| 26| 2023-07-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          7| Karan|     Pune| 33| 2023-06-22|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
+-----------+------+---------+---+-----------+



In [38]:
#32
customers.orderBy(F.col("age").desc()).show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [39]:
#33
products.orderBy(F.col("price").desc()).show(5)

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       107|  Smartphone|Electronics|40000|
|       110|      Tablet|Electronics|30000|
|       106|        Desk|  Furniture|15000|
|       104|     Monitor|Electronics|12000|
+----------+------------+-----------+-----+
only showing top 5 rows


In [40]:
#34
products.orderBy(F.col("price").asc()).show(3)

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       109|         Pen| Stationery|   20|
|       108|    Notebook| Stationery|  100|
|       103|    Keyboard|Electronics| 1500|
+----------+------------+-----------+-----+
only showing top 3 rows


In [41]:
#35
orders.orderBy("order_date").show()

+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|       1|          1|2024-03-01|Delivered|
|       2|          2|2024-03-02|Delivered|
|       3|          3|2024-03-03|Cancelled|
|       4|          4|2024-03-04|Delivered|
|       5|          5|2024-03-05|Delivered|
|       6|          6|2024-03-06|Delivered|
|       7|          7|2024-03-07|  Pending|
|       8|          8|2024-03-08|Delivered|
|       9|          9|2024-03-09|Delivered|
|      10|         10|2024-03-10|Delivered|
+--------+-----------+----------+---------+



In [42]:
#36
payments.orderBy("amount").show()

+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|        10|      10|         UPI|   200|
|         9|       9|         UPI|   500|
|         2|       2|         UPI|  1500|
|         7|       7|  Debit Card|  3000|
|         6|       6|         UPI|  9000|
|         4|       4| Credit Card| 12000|
|         8|       8| Credit Card| 30000|
|         5|       5|         UPI| 40000|
|         3|       3|  Debit Card| 75000|
|         1|       1| Credit Card| 81000|
+----------+--------+------------+------+



In [46]:
#37
payments.orderBy(F.col("amount").desc()).show(3)

+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|         1|       1| Credit Card| 81000|
|         3|       3|  Debit Card| 75000|
|         5|       5|         UPI| 40000|
+----------+--------+------------+------+
only showing top 3 rows


In [45]:
#38
payments.orderBy(F.col("amount").asc()).show(3)

+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|        10|      10|         UPI|   200|
|         9|       9|         UPI|   500|
|         2|       2|         UPI|  1500|
+----------+--------+------------+------+
only showing top 3 rows


In [44]:
#39
customers.orderBy("city").show()


+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          2| Priya|Bangalore| 32| 2023-02-12|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         15|Simran|Bangalore| 25| 2023-09-18|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|          7| Karan|     Pune| 33| 2023-06-22|
+-----------+------+---------+---+-----------+



In [43]:
#40
products.orderBy("category").show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       102|  Headphones|Electronics| 3000|
|       103|    Keyboard|Electronics| 1500|
|       104|     Monitor|Electronics|12000|
|       107|  Smartphone|Electronics|40000|
|       110|      Tablet|Electronics|30000|
|       105|Office Chair|  Furniture| 7000|
|       106|        Desk|  Furniture|15000|
|       108|    Notebook| Stationery|  100|
|       109|         Pen| Stationery|   20|
+----------+------------+-----------+-----+



**SECTION 5 — Aggregations**

In [47]:
#41
payments.agg(F.sum("amount").alias("total_revenue")).show()

+-------------+
|total_revenue|
+-------------+
|       252200|
+-------------+



In [48]:
#42
payments.agg(F.avg("amount").alias("avg_payment")).show()

+-----------+
|avg_payment|
+-----------+
|    25220.0|
+-----------+



In [49]:
#43
payments.agg(F.max("amount").alias("max_payment")).show()

+-----------+
|max_payment|
+-----------+
|      81000|
+-----------+



In [50]:
#44
payments.agg(F.min("amount").alias("min_payment")).show()

+-----------+
|min_payment|
+-----------+
|        200|
+-----------+



In [51]:
#45
customers.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    3|
|  Chennai|    2|
|   Mumbai|    3|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+



In [52]:
#46
products.groupBy("category").count().show()

+-----------+-----+
|   category|count|
+-----------+-----+
| Stationery|    2|
|Electronics|    6|
|  Furniture|    2|
+-----------+-----+



In [53]:
#47
products.groupBy("category").agg(F.avg("price").alias("avg_price")).show()

+-----------+------------------+
|   category|         avg_price|
+-----------+------------------+
| Stationery|              60.0|
|Electronics|26916.666666666668|
|  Furniture|           11000.0|
+-----------+------------------+



In [54]:
#48
order_items.groupBy("product_id").agg(F.sum("quantity").alias("total_qty")).show()

+----------+---------+
|product_id|total_qty|
+----------+---------+
|       108|        5|
|       101|        2|
|       103|        3|
|       107|        1|
|       102|        5|
|       109|       10|
|       110|        1|
|       104|        1|
+----------+---------+



In [55]:
#49
customers.agg(F.avg("age").alias("avg_age")).show()

+------------------+
|           avg_age|
+------------------+
|31.466666666666665|
+------------------+



In [56]:
#50
print(orders.count())

10


**SECTION 6 — Joins**

In [57]:
#51
orders_customers = orders.join(customers, on="customer_id", how="inner")
orders_customers.show()

+-----------+--------+----------+---------+------+---------+---+-----------+
|customer_id|order_id|order_date|   status|  name|     city|age|signup_date|
+-----------+--------+----------+---------+------+---------+---+-----------+
|          1|       1|2024-03-01|Delivered|  Amit|Hyderabad| 28| 2023-01-10|
|          2|       2|2024-03-02|Delivered| Priya|Bangalore| 32| 2023-02-12|
|          3|       3|2024-03-03|Cancelled| Rahul|   Mumbai| 29| 2023-03-14|
|          4|       4|2024-03-04|Delivered| Sneha|    Delhi| 35| 2023-04-15|
|          5|       5|2024-03-05|Delivered| Arjun|  Chennai| 27| 2023-05-11|
|          6|       6|2024-03-06|Delivered| Meera|Hyderabad| 31| 2023-06-10|
|          7|       7|2024-03-07|  Pending| Karan|     Pune| 33| 2023-06-22|
|          8|       8|2024-03-08|Delivered|  Neha|    Delhi| 30| 2023-07-10|
|          9|       9|2024-03-09|Delivered| Divya|Bangalore| 26| 2023-07-15|
|         10|      10|2024-03-10|Delivered|Vikram|   Mumbai| 40| 2023-08-01|

In [58]:
#52
orders_customers.select("name", "status").show()

+------+---------+
|  name|   status|
+------+---------+
|  Amit|Delivered|
| Priya|Delivered|
| Rahul|Cancelled|
| Sneha|Delivered|
| Arjun|Delivered|
| Meera|Delivered|
| Karan|  Pending|
|  Neha|Delivered|
| Divya|Delivered|
|Vikram|Delivered|
+------+---------+



In [59]:
#53
orders_items_joined = orders.join(order_items, on="order_id", how="inner")
orders_items_joined.show()

+--------+-----------+----------+---------+----------+--------+
|order_id|customer_id|order_date|   status|product_id|quantity|
+--------+-----------+----------+---------+----------+--------+
|       1|          1|2024-03-01|Delivered|       102|       2|
|       1|          1|2024-03-01|Delivered|       101|       1|
|       2|          2|2024-03-02|Delivered|       103|       1|
|       3|          3|2024-03-03|Cancelled|       101|       1|
|       4|          4|2024-03-04|Delivered|       104|       1|
|       5|          5|2024-03-05|Delivered|       107|       1|
|       6|          6|2024-03-06|Delivered|       102|       3|
|       7|          7|2024-03-07|  Pending|       103|       2|
|       8|          8|2024-03-08|Delivered|       110|       1|
|       9|          9|2024-03-09|Delivered|       108|       5|
|      10|         10|2024-03-10|Delivered|       109|      10|
+--------+-----------+----------+---------+----------+--------+



In [60]:
#54
items_products = order_items.join(products, on="product_id", how="inner")
items_products.show()

+----------+--------+--------+------------+-----------+-----+
|product_id|order_id|quantity|product_name|   category|price|
+----------+--------+--------+------------+-----------+-----+
|       101|       3|       1|      Laptop|Electronics|75000|
|       101|       1|       1|      Laptop|Electronics|75000|
|       102|       6|       3|  Headphones|Electronics| 3000|
|       102|       1|       2|  Headphones|Electronics| 3000|
|       103|       7|       2|    Keyboard|Electronics| 1500|
|       103|       2|       1|    Keyboard|Electronics| 1500|
|       104|       4|       1|     Monitor|Electronics|12000|
|       107|       5|       1|  Smartphone|Electronics|40000|
|       108|       9|       5|    Notebook| Stationery|  100|
|       109|      10|      10|         Pen| Stationery|   20|
|       110|       8|       1|      Tablet|Electronics|30000|
+----------+--------+--------+------------+-----------+-----+



In [61]:
#55
items_products = items_products.withColumn("revenue", F.col("quantity") * F.col("price"))
items_products.show()

+----------+--------+--------+------------+-----------+-----+-------+
|product_id|order_id|quantity|product_name|   category|price|revenue|
+----------+--------+--------+------------+-----------+-----+-------+
|       101|       3|       1|      Laptop|Electronics|75000|  75000|
|       101|       1|       1|      Laptop|Electronics|75000|  75000|
|       102|       6|       3|  Headphones|Electronics| 3000|   9000|
|       102|       1|       2|  Headphones|Electronics| 3000|   6000|
|       103|       7|       2|    Keyboard|Electronics| 1500|   3000|
|       103|       2|       1|    Keyboard|Electronics| 1500|   1500|
|       104|       4|       1|     Monitor|Electronics|12000|  12000|
|       107|       5|       1|  Smartphone|Electronics|40000|  40000|
|       108|       9|       5|    Notebook| Stationery|  100|    500|
|       109|      10|      10|         Pen| Stationery|   20|    200|
|       110|       8|       1|      Tablet|Electronics|30000|  30000|
+----------+--------

In [70]:
#56
full_join = orders .join(customers, on="customer_id", how="inner").join(order_items, on="order_id", how="inner")\
.join(products, on="product_id", how="inner")
full_join.show()

+----------+--------+-----------+----------+---------+------+---------+---+-----------+--------+------------+-----------+-----+
|product_id|order_id|customer_id|order_date|   status|  name|     city|age|signup_date|quantity|product_name|   category|price|
+----------+--------+-----------+----------+---------+------+---------+---+-----------+--------+------------+-----------+-----+
|       102|       1|          1|2024-03-01|Delivered|  Amit|Hyderabad| 28| 2023-01-10|       2|  Headphones|Electronics| 3000|
|       101|       1|          1|2024-03-01|Delivered|  Amit|Hyderabad| 28| 2023-01-10|       1|      Laptop|Electronics|75000|
|       103|       2|          2|2024-03-02|Delivered| Priya|Bangalore| 32| 2023-02-12|       1|    Keyboard|Electronics| 1500|
|       101|       3|          3|2024-03-03|Cancelled| Rahul|   Mumbai| 29| 2023-03-14|       1|      Laptop|Electronics|75000|
|       104|       4|          4|2024-03-04|Delivered| Sneha|    Delhi| 35| 2023-04-15|       1|     Mon

In [63]:
#57
full_join.select("name", "product_name", "quantity").show()

+------+------------+--------+
|  name|product_name|quantity|
+------+------------+--------+
|  Amit|  Headphones|       2|
|  Amit|      Laptop|       1|
| Priya|    Keyboard|       1|
| Rahul|      Laptop|       1|
| Sneha|     Monitor|       1|
| Arjun|  Smartphone|       1|
| Meera|  Headphones|       3|
| Karan|    Keyboard|       2|
|  Neha|      Tablet|       1|
| Divya|    Notebook|       5|
|Vikram|         Pen|      10|
+------+------------+--------+



In [64]:
#58
full_join.withColumn("revenue", F.col("quantity") * F.col("price")) \
    .groupBy("order_id").agg(F.sum("revenue").alias("order_revenue")).show()

+--------+-------------+
|order_id|order_revenue|
+--------+-------------+
|       1|        81000|
|       6|         9000|
|       3|        75000|
|       5|        40000|
|       9|          500|
|       4|        12000|
|       8|        30000|
|       7|         3000|
|      10|          200|
|       2|         1500|
+--------+-------------+



In [69]:
#59
full_join.withColumn("revenue", F.col("quantity") * F.col("price")).groupBy("product_id", "product_name")\
.agg(F.sum("revenue").alias("product_revenue")).show()


+----------+------------+---------------+
|product_id|product_name|product_revenue|
+----------+------------+---------------+
|       103|    Keyboard|           4500|
|       104|     Monitor|          12000|
|       109|         Pen|            200|
|       101|      Laptop|         150000|
|       102|  Headphones|          15000|
|       107|  Smartphone|          40000|
|       110|      Tablet|          30000|
|       108|    Notebook|            500|
+----------+------------+---------------+



In [68]:
#60
full_join.withColumn("revenue", F.col("quantity") * F.col("price"))\
.groupBy("customer_id", "name").agg(F.sum("revenue").alias("customer_revenue")).show()


+-----------+------+----------------+
|customer_id|  name|customer_revenue|
+-----------+------+----------------+
|          9| Divya|             500|
|          8|  Neha|           30000|
|          1|  Amit|           81000|
|          7| Karan|            3000|
|         10|Vikram|             200|
|          5| Arjun|           40000|
|          4| Sneha|           12000|
|          6| Meera|            9000|
|          3| Rahul|           75000|
|          2| Priya|            1500|
+-----------+------+----------------+



**SECTION 7 — GroupBy Analytics**

In [71]:
revenue_df = full_join.withColumn("revenue", F.col("quantity") * F.col("price"))


In [72]:
#61
orders.groupBy("customer_id").count().alias("order_count").show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|          1|    1|
|          6|    1|
|          3|    1|
|          5|    1|
|          9|    1|
|          4|    1|
|          8|    1|
|          7|    1|
|         10|    1|
|          2|    1|
+-----------+-----+



In [73]:
#62
orders_customers.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    2|
+---------+-----+



In [74]:
#63
revenue_df.groupBy("product_id", "product_name").agg(F.sum("revenue").alias("total_revenue")).show()

+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|       103|    Keyboard|         4500|
|       104|     Monitor|        12000|
|       109|         Pen|          200|
|       101|      Laptop|       150000|
|       102|  Headphones|        15000|
|       107|  Smartphone|        40000|
|       110|      Tablet|        30000|
|       108|    Notebook|          500|
+----------+------------+-------------+



In [75]:
#64
revenue_df.groupBy("category").agg(F.sum("revenue").alias("total_revenue")).show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
| Stationery|          700|
|Electronics|       251500|
+-----------+-------------+



In [76]:
#65
revenue_df.groupBy("city").agg(F.sum("revenue").alias("total_revenue")).show()


+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|         2000|
|  Chennai|        40000|
|   Mumbai|        75200|
|     Pune|         3000|
|    Delhi|        42000|
|Hyderabad|        90000|
+---------+-------------+



In [77]:
#66
order_items.groupBy("product_id").agg(F.sum("quantity").alias("total_qty")).show()

+----------+---------+
|product_id|total_qty|
+----------+---------+
|       108|        5|
|       101|        2|
|       103|        3|
|       107|        1|
|       102|        5|
|       109|       10|
|       110|        1|
|       104|        1|
+----------+---------+



In [78]:
#67
revenue_df.groupBy("category").agg(F.sum("quantity").alias("total_qty")).show()

+-----------+---------+
|   category|total_qty|
+-----------+---------+
| Stationery|       15|
|Electronics|       13|
+-----------+---------+



In [79]:
#68
revenue_df.groupBy("customer_id", "name").agg(F.sum("revenue").alias("total_revenue")).show()

+-----------+------+-------------+
|customer_id|  name|total_revenue|
+-----------+------+-------------+
|          9| Divya|          500|
|          8|  Neha|        30000|
|          1|  Amit|        81000|
|          7| Karan|         3000|
|         10|Vikram|          200|
|          5| Arjun|        40000|
|          4| Sneha|        12000|
|          6| Meera|         9000|
|          3| Rahul|        75000|
|          2| Priya|         1500|
+-----------+------+-------------+



In [80]:
#69
revenue_df.groupBy("customer_id", "name").agg(F.sum("revenue").alias("total_revenue")).orderBy(F.col("total_revenue").desc()).show(5)

+-----------+-----+-------------+
|customer_id| name|total_revenue|
+-----------+-----+-------------+
|          1| Amit|        81000|
|          3|Rahul|        75000|
|          5|Arjun|        40000|
|          8| Neha|        30000|
|          4|Sneha|        12000|
+-----------+-----+-------------+
only showing top 5 rows


In [81]:
#70
revenue_df.groupBy("product_id", "product_name").agg(F.sum("revenue").alias("total_revenue")).orderBy(F.col("total_revenue").desc()).show(3)

+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|       101|      Laptop|       150000|
|       107|  Smartphone|        40000|
|       110|      Tablet|        30000|
+----------+------------+-------------+
only showing top 3 rows


**SECTION 8 — Window Functions**

In [82]:
#71
w_price = Window.orderBy(F.col("price").desc())
products.withColumn("rank", F.rank().over(w_price)).show()

+----------+------------+-----------+-----+----+
|product_id|product_name|   category|price|rank|
+----------+------------+-----------+-----+----+
|       101|      Laptop|Electronics|75000|   1|
|       107|  Smartphone|Electronics|40000|   2|
|       110|      Tablet|Electronics|30000|   3|
|       106|        Desk|  Furniture|15000|   4|
|       104|     Monitor|Electronics|12000|   5|
|       105|Office Chair|  Furniture| 7000|   6|
|       102|  Headphones|Electronics| 3000|   7|
|       103|    Keyboard|Electronics| 1500|   8|
|       108|    Notebook| Stationery|  100|   9|
|       109|         Pen| Stationery|   20|  10|
+----------+------------+-----------+-----+----+



In [83]:
#72
w_cat = Window.partitionBy("category").orderBy(F.col("price").desc())
products.withColumn("rank_in_category", F.rank().over(w_cat)).show()

+----------+------------+-----------+-----+----------------+
|product_id|product_name|   category|price|rank_in_category|
+----------+------------+-----------+-----+----------------+
|       101|      Laptop|Electronics|75000|               1|
|       107|  Smartphone|Electronics|40000|               2|
|       110|      Tablet|Electronics|30000|               3|
|       104|     Monitor|Electronics|12000|               4|
|       102|  Headphones|Electronics| 3000|               5|
|       103|    Keyboard|Electronics| 1500|               6|
|       106|        Desk|  Furniture|15000|               1|
|       105|Office Chair|  Furniture| 7000|               2|
|       108|    Notebook| Stationery|  100|               1|
|       109|         Pen| Stationery|   20|               2|
+----------+------------+-----------+-----+----------------+



In [84]:
#73
w_signup = Window.orderBy("signup_date")
customers.withColumn("row_num", F.row_number().over(w_signup)).show()

+-----------+------+---------+---+-----------+-------+
|customer_id|  name|     city|age|signup_date|row_num|
+-----------+------+---------+---+-----------+-------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|      1|
|          2| Priya|Bangalore| 32| 2023-02-12|      2|
|          3| Rahul|   Mumbai| 29| 2023-03-14|      3|
|          4| Sneha|    Delhi| 35| 2023-04-15|      4|
|          5| Arjun|  Chennai| 27| 2023-05-11|      5|
|          6| Meera|Hyderabad| 31| 2023-06-10|      6|
|          7| Karan|     Pune| 33| 2023-06-22|      7|
|          8|  Neha|    Delhi| 30| 2023-07-10|      8|
|          9| Divya|Bangalore| 26| 2023-07-15|      9|
|         10|Vikram|   Mumbai| 40| 2023-08-01|     10|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|     11|
|         12|Sanjay|    Delhi| 38| 2023-08-21|     12|
|         13|Naveen|  Chennai| 28| 2023-09-01|     13|
|         14|Farhan|   Mumbai| 36| 2023-09-10|     14|
|         15|Simran|Bangalore| 25| 2023-09-18|     15|
+---------

In [85]:
#74
cust_rev = revenue_df.groupBy("customer_id", "name").agg(F.sum("revenue").alias("total_revenue"))
w_rev = Window.orderBy(F.col("total_revenue").desc())
cust_rev.withColumn("rank", F.rank().over(w_rev)).show()

+-----------+------+-------------+----+
|customer_id|  name|total_revenue|rank|
+-----------+------+-------------+----+
|          1|  Amit|        81000|   1|
|          3| Rahul|        75000|   2|
|          5| Arjun|        40000|   3|
|          8|  Neha|        30000|   4|
|          4| Sneha|        12000|   5|
|          6| Meera|         9000|   6|
|          7| Karan|         3000|   7|
|          2| Priya|         1500|   8|
|          9| Divya|          500|   9|
|         10|Vikram|          200|  10|
+-----------+------+-------------+----+



In [86]:
#75
cust_rev.withColumn("dense_rank", F.dense_rank().over(w_rev)).show()

+-----------+------+-------------+----------+
|customer_id|  name|total_revenue|dense_rank|
+-----------+------+-------------+----------+
|          1|  Amit|        81000|         1|
|          3| Rahul|        75000|         2|
|          5| Arjun|        40000|         3|
|          8|  Neha|        30000|         4|
|          4| Sneha|        12000|         5|
|          6| Meera|         9000|         6|
|          7| Karan|         3000|         7|
|          2| Priya|         1500|         8|
|          9| Divya|          500|         9|
|         10|Vikram|          200|        10|
+-----------+------+-------------+----------+



In [87]:
#76
w_city = Window.partitionBy("city").orderBy(F.col("total_revenue").desc())
cust_city_rev = revenue_df.groupBy("customer_id", "name", "city").agg(F.sum("revenue").alias("total_revenue"))
cust_city_rev.withColumn("rank", F.rank().over(w_city)).filter(F.col("rank") == 1).show()

+-----------+-----+---------+-------------+----+
|customer_id| name|     city|total_revenue|rank|
+-----------+-----+---------+-------------+----+
|          2|Priya|Bangalore|         1500|   1|
|          5|Arjun|  Chennai|        40000|   1|
|          8| Neha|    Delhi|        30000|   1|
|          1| Amit|Hyderabad|        81000|   1|
|          3|Rahul|   Mumbai|        75000|   1|
|          7|Karan|     Pune|         3000|   1|
+-----------+-----+---------+-------------+----+



In [88]:
#77
prod_cat_rev = revenue_df.groupBy("product_id", "product_name", "category").agg(F.sum("revenue").alias("total_revenue"))
w_cat_rev = Window.partitionBy("category").orderBy(F.col("total_revenue").desc())
prod_cat_rev.withColumn("rank", F.rank().over(w_cat_rev)).filter(F.col("rank") == 1).show()


+----------+------------+-----------+-------------+----+
|product_id|product_name|   category|total_revenue|rank|
+----------+------------+-----------+-------------+----+
|       101|      Laptop|Electronics|       150000|   1|
|       108|    Notebook| Stationery|          500|   1|
+----------+------------+-----------+-------------+----+



In [89]:
#78
order_date_rev = revenue_df.groupBy("order_date").agg(F.sum("revenue").alias("daily_revenue"))
w_date = Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
order_date_rev.withColumn("running_total", F.sum("daily_revenue").over(w_date)).show()

+----------+-------------+-------------+
|order_date|daily_revenue|running_total|
+----------+-------------+-------------+
|2024-03-01|        81000|        81000|
|2024-03-02|         1500|        82500|
|2024-03-03|        75000|       157500|
|2024-03-04|        12000|       169500|
|2024-03-05|        40000|       209500|
|2024-03-06|         9000|       218500|
|2024-03-07|         3000|       221500|
|2024-03-08|        30000|       251500|
|2024-03-09|          500|       252000|
|2024-03-10|          200|       252200|
+----------+-------------+-------------+



In [90]:
#79
prod_rev = revenue_df.groupBy("product_id", "product_name").agg(F.sum("revenue").alias("total_revenue"))
w_prod = Window.orderBy("product_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
prod_rev.withColumn("running_total", F.sum("total_revenue").over(w_prod)).show()

+----------+------------+-------------+-------------+
|product_id|product_name|total_revenue|running_total|
+----------+------------+-------------+-------------+
|       101|      Laptop|       150000|       150000|
|       102|  Headphones|        15000|       165000|
|       103|    Keyboard|         4500|       169500|
|       104|     Monitor|        12000|       181500|
|       107|  Smartphone|        40000|       221500|
|       108|    Notebook|          500|       222000|
|       109|         Pen|          200|       222200|
|       110|      Tablet|        30000|       252200|
+----------+------------+-------------+-------------+



In [91]:
#80
w_pay = Window.orderBy(F.col("amount").desc())
payments.withColumn("rank", F.rank().over(w_pay)).show()


+----------+--------+------------+------+----+
|payment_id|order_id|payment_type|amount|rank|
+----------+--------+------------+------+----+
|         1|       1| Credit Card| 81000|   1|
|         3|       3|  Debit Card| 75000|   2|
|         5|       5|         UPI| 40000|   3|
|         8|       8| Credit Card| 30000|   4|
|         4|       4| Credit Card| 12000|   5|
|         6|       6|         UPI|  9000|   6|
|         7|       7|  Debit Card|  3000|   7|
|         2|       2|         UPI|  1500|   8|
|         9|       9|         UPI|   500|   9|
|        10|      10|         UPI|   200|  10|
+----------+--------+------------+------+----+



**SECTION 9 — RDD Exercises**

In [93]:
logs = sc.textFile("logs.txt")


In [95]:
#81
print(logs.count())

20


In [96]:
#82
usernames = logs.map(lambda line: line.split()[1])
print(usernames.collect())

['Amit', 'Priya', 'Rahul', 'Amit', 'Sneha', 'Amit', 'Priya', 'Arjun', 'Sneha', 'Rahul', 'Amit', 'Neha', 'Neha', 'Vikram', 'Vikram', 'Farhan', 'Farhan', 'Farhan', 'Simran', 'Simran']


In [97]:
#83
actions = logs.map(lambda line: line.split()[0])
print(actions.collect())

['login', 'login', 'view', 'purchase', 'view', 'login', 'purchase', 'view', 'purchase', 'login', 'logout', 'login', 'purchase', 'view', 'purchase', 'login', 'view', 'purchase', 'login', 'purchase']


In [98]:
#84
unique_users = logs.map(lambda line: line.split()[1]).distinct()
print(unique_users.collect())

['Amit', 'Vikram', 'Simran', 'Priya', 'Rahul', 'Sneha', 'Arjun', 'Neha', 'Farhan']


In [99]:
#85
login_count = logs.filter(lambda line: line.startswith("login")).count()
print(login_count)

7


In [100]:
#86
purchase_count = logs.filter(lambda line: line.startswith("purchase")).count()
print(purchase_count)

7


In [ ]:
#87
view_count = logs.filter(lambda line: line.startswith("view")).count()
print(view_count)


In [101]:
#88
user_pairs = logs.map(lambda line: (line.split()[1], 1))
print(user_pairs.collect())

[('Amit', 1), ('Priya', 1), ('Rahul', 1), ('Amit', 1), ('Sneha', 1), ('Amit', 1), ('Priya', 1), ('Arjun', 1), ('Sneha', 1), ('Rahul', 1), ('Amit', 1), ('Neha', 1), ('Neha', 1), ('Vikram', 1), ('Vikram', 1), ('Farhan', 1), ('Farhan', 1), ('Farhan', 1), ('Simran', 1), ('Simran', 1)]


In [102]:
#89
user_activity = user_pairs.reduceByKey(lambda a, b: a + b)
print(user_activity.collect())

[('Amit', 4), ('Vikram', 2), ('Simran', 2), ('Priya', 2), ('Rahul', 2), ('Sneha', 2), ('Arjun', 1), ('Neha', 2), ('Farhan', 3)]


In [103]:
#90
most_active = user_activity.sortBy(lambda x: x[1], ascending=False).first()
print(most_active)

('Amit', 4)


**SECTION 10 — Spark SQL**

In [104]:
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
products.createOrReplaceTempView("products")
payments.createOrReplaceTempView("payments")
order_items.createOrReplaceTempView("order_items")

In [105]:
#91
spark.sql("SELECT * FROM customers").show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [107]:
#92
spark.sql("SELECT * FROM customers WHERE age > 30").show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          2| Priya|Bangalore| 32| 2023-02-12|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
+-----------+------+---------+---+-----------+



In [108]:
#93
spark.sql("SELECT city, COUNT(*) AS customer_count FROM customers GROUP BY city").show()

+---------+--------------+
|     city|customer_count|
+---------+--------------+
|Bangalore|             3|
|  Chennai|             2|
|   Mumbai|             3|
|     Pune|             1|
|    Delhi|             3|
|Hyderabad|             3|
+---------+--------------+



In [109]:
#94
spark.sql("SELECT city, AVG(age) AS avg_age FROM customers GROUP BY city").show()

+---------+------------------+
|     city|           avg_age|
+---------+------------------+
|Bangalore|27.666666666666668|
|  Chennai|              27.5|
|   Mumbai|              35.0|
|     Pune|              33.0|
|    Delhi|34.333333333333336|
|Hyderabad|              31.0|
+---------+------------------+



In [106]:
#95
spark.sql("""
    SELECT c.name, o.order_id, o.order_date, o.status
    FROM customers c JOIN orders o ON c.customer_id = o.customer_id
""").show()

+------+--------+----------+---------+
|  name|order_id|order_date|   status|
+------+--------+----------+---------+
|  Amit|       1|2024-03-01|Delivered|
| Priya|       2|2024-03-02|Delivered|
| Rahul|       3|2024-03-03|Cancelled|
| Sneha|       4|2024-03-04|Delivered|
| Arjun|       5|2024-03-05|Delivered|
| Meera|       6|2024-03-06|Delivered|
| Karan|       7|2024-03-07|  Pending|
|  Neha|       8|2024-03-08|Delivered|
| Divya|       9|2024-03-09|Delivered|
|Vikram|      10|2024-03-10|Delivered|
+------+--------+----------+---------+



In [110]:
#96
spark.sql("""
    SELECT c.name, COUNT(o.order_id) AS total_orders
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.name
""").show()

+------+------------+
|  name|total_orders|
+------+------------+
| Divya|           1|
| Meera|           1|
| Sneha|           1|
| Priya|           1|
|Vikram|           1|
| Rahul|           1|
| Arjun|           1|
|  Amit|           1|
|  Neha|           1|
| Karan|           1|
+------+------------+



In [111]:
#97
spark.sql("""
    SELECT oi.order_id, p.product_name, oi.quantity, p.price
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
""").show()

+--------+------------+--------+-----+
|order_id|product_name|quantity|price|
+--------+------------+--------+-----+
|       3|      Laptop|       1|75000|
|       1|      Laptop|       1|75000|
|       6|  Headphones|       3| 3000|
|       1|  Headphones|       2| 3000|
|       7|    Keyboard|       2| 1500|
|       2|    Keyboard|       1| 1500|
|       4|     Monitor|       1|12000|
|       5|  Smartphone|       1|40000|
|       9|    Notebook|       5|  100|
|      10|         Pen|      10|   20|
|       8|      Tablet|       1|30000|
+--------+------------+--------+-----+



In [112]:
#98
spark.sql("""
    SELECT oi.order_id, p.product_name, oi.quantity * p.price AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
""").show()

+--------+------------+-------+
|order_id|product_name|revenue|
+--------+------------+-------+
|       3|      Laptop|  75000|
|       1|      Laptop|  75000|
|       6|  Headphones|   9000|
|       1|  Headphones|   6000|
|       7|    Keyboard|   3000|
|       2|    Keyboard|   1500|
|       4|     Monitor|  12000|
|       5|  Smartphone|  40000|
|       9|    Notebook|    500|
|      10|         Pen|    200|
|       8|      Tablet|  30000|
+--------+------------+-------+



In [113]:
#99
spark.sql("""
    SELECT c.name, SUM(oi.quantity * p.price) AS total_revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY c.name
    ORDER BY total_revenue DESC
    LIMIT 3
""").show()

+-----+-------------+
| name|total_revenue|
+-----+-------------+
| Amit|        81000|
|Rahul|        75000|
|Arjun|        40000|
+-----+-------------+



In [114]:
#100
spark.sql("""
    SELECT p.product_name, SUM(oi.quantity * p.price) AS total_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.product_name
    ORDER BY total_revenue DESC
    LIMIT 1
""").show()

+------------+-------------+
|product_name|total_revenue|
+------------+-------------+
|      Laptop|       150000|
+------------+-------------+



** FINAL CAPSTONE MINI PROJECT**

In [115]:
#1
final_df = orders \
    .join(customers,   on="customer_id", how="inner") \
    .join(order_items, on="order_id",    how="inner") \
    .join(products,    on="product_id",  how="inner") \
    .join(payments,    on="order_id",    how="left")


In [119]:
# 2
final_df = final_df.withColumn("revenue", F.col("quantity") * F.col("price"))

In [121]:
# 3
print("Top Customers")
final_df.groupBy("customer_id", "name") \
    .agg(F.sum("revenue").alias("total_revenue")) \
    .orderBy(F.col("total_revenue").desc()).show(5)

Top Customers
+-----------+-----+-------------+
|customer_id| name|total_revenue|
+-----------+-----+-------------+
|          1| Amit|        81000|
|          3|Rahul|        75000|
|          5|Arjun|        40000|
|          8| Neha|        30000|
|          4|Sneha|        12000|
+-----------+-----+-------------+
only showing top 5 rows


In [123]:
# 4
print("Top Products ")
final_df.groupBy("product_id", "product_name").agg(F.sum("revenue").alias("total_revenue")).orderBy(F.col("total_revenue").desc()).show(5)

Top Products 
+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|       101|      Laptop|       150000|
|       107|  Smartphone|        40000|
|       110|      Tablet|        30000|
|       102|  Headphones|        15000|
|       104|     Monitor|        12000|
+----------+------------+-------------+
only showing top 5 rows


In [124]:
# 5
print("Revenue per City")
final_df.groupBy("city").agg(F.sum("revenue").alias("total_revenue")).orderBy(F.col("total_revenue").desc()).show()


Revenue per City
+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Hyderabad|        90000|
|   Mumbai|        75200|
|    Delhi|        42000|
|  Chennai|        40000|
|     Pune|         3000|
|Bangalore|         2000|
+---------+-------------+



In [125]:
# 6
cust_rev_final = final_df.groupBy("customer_id", "name").agg(F.sum("revenue").alias("total_revenue"))
w_final = Window.orderBy(F.col("total_revenue").desc())
print("Customer Revenue Ranking")
cust_rev_final.withColumn("rank", F.rank().over(w_final)).show()


Customer Revenue Ranking
+-----------+------+-------------+----+
|customer_id|  name|total_revenue|rank|
+-----------+------+-------------+----+
|          1|  Amit|        81000|   1|
|          3| Rahul|        75000|   2|
|          5| Arjun|        40000|   3|
|          8|  Neha|        30000|   4|
|          4| Sneha|        12000|   5|
|          6| Meera|         9000|   6|
|          7| Karan|         3000|   7|
|          2| Priya|         1500|   8|
|          9| Divya|          500|   9|
|         10|Vikram|          200|  10|
+-----------+------+-------------+----+



In [126]:
# 7
final_df.write.mode("overwrite").csv("ecommerce_report", header=True)
print("Final dataset saved to ecommerce_report/")

Final dataset saved to ecommerce_report/
